# Fresh AoU VaDE optimization with four clusters

This notebook includes preprocessing, optimization, and export helpers inline. It still requires the JHU model modules in `MODULE_DIR`, the AoU feature CSV, and JHU metadata Parquet. Review the paths in the settings cell before running.

The search retains the original preprocessing, 80/20 split, fixed four clusters, search space, schedule, and two objectives: minimize held-out VaDE loss and maximize silhouette × active clusters / 4. This held-out split is used for tuning and early stopping, not an independent final test set. Cluster quality is evaluated over the full cohort.

Outputs are isolated under `../outputs/<new-study-name>/`: study database, run settings, preprocessing metadata, mapping and feature lists, sweep/Pareto CSVs, plots, and `trial-<N>/model.pt`.

In [ ]:
import importlib
import json
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

## Settings and paths

In [ ]:
MODULE_DIR = (Path.home() / "workspace/rw-migration-aou-rw-71e4e7ad/notebooks/s").resolve()
FEATURE_MATRIX_PATH = Path("feature_matrix_wide_deaths.csv").resolve()
METADATA_PATH = Path("meta_HFCohort_deid.parquet").resolve()
OUTPUT_ROOT = Path("../outputs").resolve()

seed = 42
N_TRIALS = 32
FIXED_C = 4
PROJECT = "vade-heartfailure-aou"
DATASET = "heartfailure-aou"
HIDDEN_DIMS = {
    "256x128x64x32": [256, 128, 64, 32],
    "256x128x64": [256, 128, 64],
    "128x64x32": [128, 64, 32],
}
SCHEDULE = {
    "gamma": 1e-3, "gamma_steps": 3000, "gmm_steps": 20,
    "beta_steps": 500, "static_steps": 500, "max_steps": 16000,
    "early_stopping_metric": "mse", "early_stopping_patience": 4,
}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
STUDY_NAME = f"s3vdc-vade-aou-fixedc4-deaths-{RUN_ID}"
EXPORT_DIR = OUTPUT_ROOT / STUDY_NAME
DB_PATH = EXPORT_DIR / "study.db"
STORAGE = f"sqlite:///{DB_PATH.as_posix()}"
PLOT_DIR = EXPORT_DIR / "plots"

required_modules = ["config", "data", "inference", "scoring", "train", "vade"]
missing = [MODULE_DIR / f"{name}.py" for name in required_modules
           if not (MODULE_DIR / f"{name}.py").is_file()]
missing += [p for p in (FEATURE_MATRIX_PATH, METADATA_PATH) if not p.is_file()]
if missing:
    raise FileNotFoundError("Update the settings paths. Missing:\n" + "\n".join(map(str, missing)))
sys.path.insert(0, str(MODULE_DIR))
for name in list(sys.modules):
    if name.split(".")[0] in required_modules:
        del sys.modules[name]
importlib.invalidate_caches()
from config import DataConfig, ScheduleConfig, TrainConfig, VaDEConfig
from data import make_dataloader, positive_weights
from inference import predict_clusters
from scoring import cluster_quality, clustpy_vade_loss
from train import evaluate, train
from vade import VaDE, load_checkpoint, save_checkpoint

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
settings = {
    "study": STUDY_NAME, "seed": seed, "n_trials": N_TRIALS,
    "fixed_c": FIXED_C, "schedule": SCHEDULE, "hidden_dims": HIDDEN_DIMS,
    "module_dir": str(MODULE_DIR), "feature_matrix": str(FEATURE_MATRIX_PATH),
    "metadata": str(METADATA_PATH), "device": str(device),
}
(EXPORT_DIR / "run_settings.json").write_text(json.dumps(settings, indent=2))
print(f"New study: {STUDY_NAME}\nDevice: {device}\nOutputs: {EXPORT_DIR}")

## Preprocessing

In [ ]:
def fit_lab_scaler(train_df):
    """Fit robust-scaling statistics (median/IQR) for non-binary columns.

    Args:
        train_df: feature DataFrame restricted to training patients.

    Returns:
        dict with "cols" (numeric column names), "median", "iqr" (Series),
        fit using training patients only.
    """
    numeric_cols = [
        col for col in train_df.columns
        if not set(train_df[col].dropna().unique()).issubset({0, 1, 0.0, 1.0})
    ]
    median = train_df[numeric_cols].median()
    iqr = (
        train_df[numeric_cols].quantile(0.75)
        - train_df[numeric_cols].quantile(0.25)
    ).replace(0, 1.0)
    return {"cols": numeric_cols, "median": median, "iqr": iqr}


def encode_measurements(df, scaler, clip=4.0):
    """Robustly z-score continuous measurements; keep binary features unchanged.

    Missing continuous values become 0, with mask=0 so they are excluded from
    reconstruction loss. Binary 0 means absent, not missing, so it stays mask=1.

    Args:
        df: full feature DataFrame (all patients), covariate-name columns.
        scaler: dict from fit_lab_scaler.
        clip: symmetric clip bound applied to z-scores.

    Returns:
        (feats, mask) DataFrames, same index as df.
    """
    numeric_cols = scaler["cols"]
    median = scaler["median"]
    iqr = scaler["iqr"]

    numeric = df[numeric_cols]
    observed_numeric = numeric.notna()

    numeric_scaled = (
        ((numeric - median) / iqr)
        .clip(-clip, clip)
        .where(observed_numeric, 0.0)
    )
    numeric_scaled.columns = [f"prof_{col}" for col in numeric_cols]

    binary = df.drop(columns=numeric_cols)
    feats = pd.concat([binary, numeric_scaled], axis=1)
    mask = pd.concat(
        [
            pd.DataFrame(1, index=df.index, columns=binary.columns),
            observed_numeric.astype(int).set_axis(numeric_scaled.columns, axis=1),
        ],
        axis=1,
    )[feats.columns]
    return feats, mask


def remap_covariate_names(col_names, covariate_map):
    """Map raw covariate_id columns to human-readable, underscore-safe names.

    Args:
        col_names: iterable of raw column labels (covariate_id-like).
        covariate_map: dict from int(covariateId) to covariateName.

    Returns:
        list of remapped, whitespace-safe names, same order as col_names.
    """
    remapped = []
    for col in col_names:
        try:
            name = covariate_map.get(int(float(col)), str(col))
        except (ValueError, TypeError):
            name = str(col)
        if ":" in name:
            name = name.split(":", 1)[1]
        name = name.strip().replace(" ", "_")
        remapped.append(name)
    return remapped


def build_aou_preprocessed_data(feature_matrix_path, metadata_path, seed):
    """
    Reproduces feats_df/mask_df/X_in/band_rate/binary_idx/num_idx/pos_weight
    exactly as the de-novo notebook does, so this optimization search tunes
    against the same feature matrix the reported de-novo model was trained on.

    Args:
        feature_matrix_path: path to feature_matrix_wide.csv.
        metadata_path: path to meta_HFCohort_deid.parquet (covariate_id -> name).
        seed: random seed for the train/test split (must match the reported
            de-novo run's seed to reproduce the same ids_train/ids_test).

    Returns:
        dict with keys: X_in, X_train, X_test, M, M_train, M_test, band_rate,
        binary_idx, num_idx, pw, feats, index, is_train.
    """
    # load + remap covariate names
    features_df = pd.read_csv(feature_matrix_path)
    features_df = features_df.set_index("person_id")
    features_df = features_df.loc[:, ~features_df.columns.duplicated()].copy()
    patient_ids = features_df.index
    if not patient_ids.is_unique or patient_ids.hasnans:
        raise ValueError("person_id must be unique and nonmissing")

    metadata = pd.read_parquet(metadata_path)
    covariate_map = metadata.set_index("covariateId")["covariateName"].to_dict()
    remapped_names = remap_covariate_names(features_df.columns, covariate_map)
    pd.DataFrame({
        "covariate_id": features_df.columns,
        "remapped_name": remapped_names,
    }).to_csv(EXPORT_DIR / "covariate_id_name_map.csv", index=False)
    features_df.columns = remapped_names

    # dedupe post-remap, train/test split, fit scaler 
    features_df = features_df.loc[:, ~features_df.columns.duplicated()].copy()
    patient_ids = features_df.index

    ids_train, ids_test = train_test_split(
        patient_ids, test_size=0.20, random_state=seed,
    )

    scaler = fit_lab_scaler(features_df.loc[ids_train])
    feats_df, mask_df = encode_measurements(features_df, scaler, clip=4.0)

    binary_cols = [col for col in feats_df.columns if not col.startswith("prof_")]
    keep_binary = feats_df[binary_cols].sum(axis=0) >= 10
    feats_df = feats_df.loc[
        :,
        list(keep_binary[keep_binary].index)
        + [col for col in feats_df.columns if col.startswith("prof_")]
    ]
    mask_df = mask_df.loc[:, feats_df.columns]

    nonzero = (feats_df != 0).any(axis=0)
    feats_df = feats_df.loc[:, nonzero]
    mask_df = mask_df.loc[:, feats_df.columns]

    X = feats_df.to_numpy(dtype="float32")
    M = mask_df.to_numpy(dtype="float32")

    band_rate = (
        feats_df.loc[ids_train]
        .where(mask_df.loc[ids_train] == 1)
        .mean()
        .fillna(0.0)
        .to_numpy(dtype="float32")
    )
    X_in = np.where(M == 1, X, band_rate).astype("float32")

    is_train = patient_ids.isin(ids_train)
    X_train, M_train = X_in[is_train], M[is_train]
    X_test, M_test = X_in[~is_train], M[~is_train]

    binary_idx = np.where(np.all((X_train == 0) | (X_train == 1), axis=0))[0].tolist()
    num_idx = np.where(~np.all((X_train == 0) | (X_train == 1), axis=0))[0].tolist()

    pw = positive_weights(X_train, M_train, binary_idx)

    return {
        "X_in": X_in, "X_train": X_train, "X_test": X_test,
        "M": M, "M_train": M_train, "M_test": M_test,
        "band_rate": band_rate, "binary_idx": binary_idx, "num_idx": num_idx,
        "pw": pw, "feats": feats_df, "mask": mask_df, "scaler": scaler,
        "index": patient_ids, "is_train": is_train,
    }

In [ ]:
P = build_aou_preprocessed_data(FEATURE_MATRIX_PATH, METADATA_PATH, seed)
I_DIM = P["X_in"].shape[1]
if I_DIM == 0 or not np.isfinite(P["X_in"]).all():
    raise ValueError("Prepared inputs are empty or contain nonfinite values; inspect missing binary values and scaling.")
TEST_LOADER = make_dataloader(P["X_test"], P["M_test"], batch_size=256, shuffle=False, drop_last=False)
FULL_LOADER = make_dataloader(P["X_in"], P["M"], batch_size=256, shuffle=False, drop_last=False)
scaler = P["scaler"]
preprocessing = {
    "seed": seed, "feature_names": list(P["feats"].columns),
    "numeric_columns": scaler["cols"],
    "median": scaler["median"].to_dict(), "iqr": scaler["iqr"].to_dict(),
    "clip": 4.0, "band_rate": P["band_rate"].tolist(),
    "train_ids": P["index"][P["is_train"]].tolist(),
    "evaluation_ids": P["index"][~P["is_train"]].tolist(),
}
(EXPORT_DIR / "preprocessing.json").write_text(json.dumps(preprocessing, indent=2, default=str))
pd.DataFrame({"encoded_feature": P["feats"].columns}).to_csv(EXPORT_DIR / "encoded_features.csv", index=False)
pd.DataFrame({"remapped_name": [c[5:] if c.startswith("prof_") else c for c in P["feats"].columns]}).to_csv(EXPORT_DIR / "vade_trained_features.csv", index=False)
print(f"Patients: {len(P['index'])}; features: {I_DIM}; train: {P['is_train'].sum()}; evaluation: {(~P['is_train']).sum()}")

## Trial functions

Training randomness is reset to seed 42 per trial.

In [ ]:
def objective(trial):
    """One trial: sample a configuration, train it, score both objectives."""
    params = suggest_params(trial)
    model, steps = fit(params)

    metrics, _, labels = evaluate(model, FULL_LOADER, device=device)
    val_loss = clustpy_vade_loss(model, TEST_LOADER, device=device)
    quality = cluster_quality(metrics, params["c"])
    sizes = np.bincount(labels.detach().cpu().numpy(), minlength=params["c"]).tolist()

    log_trial(trial, params, model,
              {**metrics, "val_loss": val_loss, "quality": quality,
               "steps_run": steps, "cluster_sizes": sizes})

    print(f"trial {trial.number:>3}  c={params['c']} z={params['z_dim']:>2} "
          f"lr={params['learning_rate']:.1e} -> val_loss={val_loss:9.2f} "
          f"quality={quality:+.4f} sil={metrics['silhouette']:+.4f} sizes={sizes}")

    return val_loss, quality


def suggest_params(trial):
    """Sample one configuration from the search space."""
    return {
        **SCHEDULE,
        "c": FIXED_C,
        "z_dim": trial.suggest_categorical("z_dim", [8, 10, 16, 20, 30]),
        "hidden_dims": HIDDEN_DIMS[trial.suggest_categorical("hidden_dims", list(HIDDEN_DIMS))],
        "batch_size": trial.suggest_categorical("batch_size", [128, 256]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "weight_decay": trial.suggest_categorical("weight_decay", [0.0, 1e-6, 1e-4, 1e-3]),
        "w_numeric": trial.suggest_float("w_numeric", 0.2, 2.0),
        "dropout_rate": trial.suggest_categorical("dropout_rate", [0.0, 0.1, 0.2]),
        "responsibility_scale": trial.suggest_categorical("responsibility_scale", [20.0, 40.0, 50.0]),
    }


def fit(params):
    """Build and train one VaDE from a sampled configuration."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    cfg = VaDEConfig.from_dict({**params, "i_dim": I_DIM})
    train_cfg = TrainConfig.from_dict({**params, "log_every": 10**9, "eval_every": 1000,
                                       "hist_every": 10**9, "embedding_every": 10**9})
    sched_cfg = ScheduleConfig.from_dict(params)

    model = VaDE(cfg, P["binary_idx"], P["num_idx"], pos_weight=P["pw"])
    history = train(model,
                    make_dataloader(P["X_train"], P["M_train"], batch_size=params["batch_size"]),
                    train_cfg, sched_cfg, device=device, eval_loader=TEST_LOADER)

    return model, history[-1]["step"] + 1

def log_trial(trial, params, model, record):
    for key, value in record.items():
        if isinstance(value, np.generic):
            value = value.item()
        trial.set_user_attr(key, value)
    folder = EXPORT_DIR / f"trial-{trial.number}"
    folder.mkdir(parents=True, exist_ok=True)
    save_checkpoint(model, folder / "model.pt",
                    band_rate=P["band_rate"], feature_names=list(P["feats"].columns),
                    preprocessing=preprocessing, study=STUDY_NAME,
                    trial=trial.number, seed=seed)

## Create the new study and optimize

In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME, storage=STORAGE,
    directions=["minimize", "maximize"],
    sampler=optuna.samplers.TPESampler(seed=seed, multivariate=True),
    load_if_exists=False,
)
print(f"Created empty study: {study.study_name}")

In [ ]:
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, gc_after_trial=True)
print(f"Completed trials: {sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)}")

## Results and Pareto front

In [ ]:
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if not completed:
    raise RuntimeError("No completed trials. Inspect the optimization output before continuing.")
results = study.trials_dataframe(attrs=("number", "values", "params", "user_attrs", "state"))
results = results.rename(columns={"values_0": "val_loss", "values_1": "quality"})
results.to_csv(EXPORT_DIR / "sweep_results_aou.csv", index=False)
pareto_df = pd.DataFrame([
    {"trial": t.number, "val_loss": t.values[0], "quality": t.values[1],
     "silhouette": t.user_attrs.get("silhouette"),
     "cluster_sizes": t.user_attrs.get("cluster_sizes"),
     "active_clusters": t.user_attrs.get("active_clusters"), **t.params}
    for t in study.best_trials
]).sort_values("val_loss").reset_index(drop=True)
pareto_df.to_csv(EXPORT_DIR / "pareto_front_full.csv", index=False)
report = pareto_df[["trial", "val_loss", "silhouette", "cluster_sizes", "z_dim"]]
report.to_csv(EXPORT_DIR / "pareto_front_report.csv", index=False)
display(report)
print("Inspect cluster sizes as well as silhouette: very small clusters can inflate apparent separation.")

In [ ]:
ax = optuna.visualization.matplotlib.plot_pareto_front(
    study, target_names=["Held-out VaDE loss", "Cluster quality"])
plt.tight_layout()
plt.savefig(PLOT_DIR / "pareto_front.svg", bbox_inches="tight")
plt.show()
if len(completed) >= 2:
    for objective_index, label in [(0, "Held-out loss"), (1, "Cluster quality")]:
        try:
            optuna.visualization.matplotlib.plot_param_importances(
                study, target=lambda t, i=objective_index: t.values[i], target_name=label)
            plt.tight_layout()
            plt.savefig(PLOT_DIR / f"parameter_importance_{objective_index}.svg", bbox_inches="tight")
            plt.show()
        except (ValueError, RuntimeError) as exc:
            print(f"Parameter importance unavailable for {label}: {exc}")

## Export selected trials

In [ ]:
def is_binary_feature(x):
    """Whether every observed value of a feature is 0 or 1.

    Args:
        x: feature Series.

    Returns:
        True for binary features, False for continuous ones such as the
        z-scored prof_ labs.
    """
    u = pd.unique(x.dropna())
    return len(u) > 0 and set(np.asarray(u).tolist()) <= {0, 1, 0.0, 1.0}


def cluster_enrichment(assignments, features):
    """Per-cluster feature effects: log odds ratio for binary, SMD for continuous.

    Binary features use a continuity-corrected log odds ratio of membership
    against the rest of the cohort. Continuous features use a standardised mean
    difference against the whole-cohort standard deviation, because counting
    values equal to 1 is meaningless for a z-scored lab. The two are put on one
    scale as Cohen's d, converting log-OR by sqrt(3)/pi.

    Args:
        assignments: frame with a "cluster" column, indexed like features.
        features: entities x features DataFrame of encoded values.

    Returns:
        Long DataFrame with cluster, Feature, feature_type, Cohens_d, Effect,
        Log_OR, SMD and the cluster/cohort prevalence and mean columns, sorted
        by cluster then descending |Cohen's d|.
    """
    df = features.loc[assignments.index]
    cluster = assignments["cluster"]
    n_total = len(df)
    scale = np.sqrt(3) / np.pi

    rows = []
    for feature in df.columns:
        v = df[feature]
        binary = is_binary_feature(v)
        sd = v.std()
        ones = (v == 1).values
        zeros = (v == 0).values

        cohort_n = int(ones.sum())
        cohort_mean = float(v.mean())

        for k in sorted(cluster.unique()):
            incl = (cluster == k).values
            n_k = int(incl.sum())

            if binary:
                a = float((ones & incl).sum()) + 0.5
                b = float((zeros & incl).sum()) + 0.5
                c_out = float((ones & ~incl).sum()) + 0.5
                d_out = float((zeros & ~incl).sum()) + 0.5
                effect = float(np.log((a / b) / (c_out / d_out)))
                cohens_d, log_or, smd = effect * scale, effect, np.nan
            else:
                if pd.isna(sd) or sd == 0:
                    effect = 0.0
                else:
                    effect = float((v[incl].mean() - v[~incl].mean()) / sd)
                cohens_d, log_or, smd = effect, np.nan, effect

            rows.append({
                "cluster": k,
                "Feature": feature,
                "feature_type": "binary" if binary else "continuous",
                "Cohens_d": cohens_d,
                "Effect": effect,
                "Log_OR": log_or,
                "SMD": smd,
                "cluster_n": int((ones & incl).sum()),
                "cluster_pct": 100.0 * float((ones & incl).sum()) / n_k if n_k else np.nan,
                "cluster_mean": float(v[incl].mean()) if n_k else np.nan,
                "Cohort_n": cohort_n,
                "Cohort_pct": 100.0 * cohort_n / n_total,
                "Cohort_mean": cohort_mean,
            })

    out = pd.DataFrame(rows)
    out["abs_d"] = out["Cohens_d"].abs()
    return out.sort_values(["cluster", "abs_d"], ascending=[True, False]).drop(columns="abs_d")

In [ ]:
def params_from_trial(trial):
    return {**SCHEDULE, "c": FIXED_C, **trial.params,
            "hidden_dims": HIDDEN_DIMS[trial.params["hidden_dims"]]}

def export_trial(trial_number):
    matches = [t for t in study.trials if t.number == trial_number]
    if not matches or matches[0].state != optuna.trial.TrialState.COMPLETE:
        raise ValueError(f"Trial {trial_number} does not exist or is not COMPLETE in this study.")
    trial = matches[0]
    folder = EXPORT_DIR / f"trial-{trial_number}"
    checkpoint = folder / "model.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Original fitted checkpoint is missing: {checkpoint}")
    model, payload = load_checkpoint(checkpoint, device=device)
    if payload.get("study") != STUDY_NAME or payload.get("trial") != trial_number:
        raise ValueError("Checkpoint does not belong to the selected study and trial")
    if payload.get("feature_names") != list(P["feats"].columns):
        raise ValueError("Checkpoint feature order differs from the prepared data")
    assignments = predict_clusters(model, P["X_in"], index=P["index"], device=device)
    assignments["split"] = np.where(P["is_train"], "train", "evaluation")
    assignments.index.name = "person_id"
    assignments.reset_index().to_parquet(folder / "assignments.parquet", index=False)
    assignments.reset_index().to_csv(folder / "assignments.csv", index=False)
    observed_features = P["feats"].where(P["mask"] == 1)
    enrichment = cluster_enrichment(assignments, observed_features)
    enrichment.to_csv(folder / "cluster_enrichment.tsv", sep="\t", index=False, na_rep="")
    counts = assignments["cluster"].value_counts().sort_index().to_dict()
    metrics = {"study": STUDY_NAME, "trial": trial_number, "weights_source": "checkpoint",
               "recorded_val_loss": trial.values[0], "recorded_quality": trial.values[1],
               "recorded_metrics": trial.user_attrs,
               "exported_cluster_sizes": {str(k): int(v) for k, v in counts.items()},
               "params": params_from_trial(trial)}
    (folder / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))
    print(f"Trial {trial_number}: {counts}\nAssignments: {folder / 'assignments.csv'}")
    return folder

In [ ]:
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
)

pareto_df = pd.DataFrame([
    {
        "trial": t.number,
        "val_loss": t.values[0],
        "quality": t.values[1],
        "silhouette": t.user_attrs.get("silhouette"),
        "cluster_sizes": t.user_attrs.get("cluster_sizes"),
        **t.params,
    }
    for t in study.best_trials
]).sort_values("val_loss").reset_index(drop=True)

display(
    pareto_df[
        ["trial", "val_loss", "silhouette", "cluster_sizes", "z_dim"]
    ]
)

CHOSEN = 19
EXPORT_ALL_PARETO = False

if CHOSEN is not None:
    if not isinstance(CHOSEN, int) or isinstance(CHOSEN, bool):
        raise ValueError("CHOSEN must be an integer trial number or None")

    chosen_trial = next(
        (t for t in study.trials if t.number == CHOSEN), None
    )
    if chosen_trial is None:
        raise ValueError(f"Trial {CHOSEN} does not exist in this study")
    if chosen_trial.state != optuna.trial.TrialState.COMPLETE:
        raise ValueError(f"Trial {CHOSEN} is not COMPLETE")

selected = (
    set(pareto_df["trial"].astype(int))
    if EXPORT_ALL_PARETO else set()
)
if CHOSEN is not None:
    selected.add(CHOSEN)

folders = {n: export_trial(n) for n in sorted(selected)}

if CHOSEN is not None:
    if CHOSEN not in set(pareto_df["trial"]):
        print("Selected trial is not on the current Pareto front.")

    config_path = EXPORT_DIR / f"best_config_trial{CHOSEN}.json"
    config_path.write_text(json.dumps(
        {
            "study": STUDY_NAME,
            "trial": CHOSEN,
            "config": params_from_trial(chosen_trial),
        },
        indent=2,
    ))

    assignments = pd.read_csv(folders[CHOSEN] / "assignments.csv")
    print("\nExported cluster sizes:")
    print(assignments["cluster"].value_counts().sort_index())
    print(f"\nAssignments: {folders[CHOSEN] / 'assignments.csv'}")
    print(f"Configuration: {config_path}")
else:
    print("Set CHOSEN to a completed trial number to export assignments.")